# Importation des librairies

In [ ]:
import numpy as np
import pandas as pd
import sklearn 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.ensemble import IsolationForest
from sklearn import neighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
from pylab import rcParams

# Génération de données

In [ ]:
X , y = sklearn.datasets.make_circles(n_samples= 400, noise= 0.1, factor= 0.3)

In [ ]:
#  Ajout des points abberants

x_dirty = np.array([[24.9, 0.1],
                    [23, 1],
                    [24.8, 0.6],
                    [27, 0.1]])

# Ajout aux données
X_dirty= np.append(X, x_dirty, axis=0)
y_dirty = np.append(y , [1, 1, 1, 1], axis=0)

In [ ]:
X_dirty
y_dirty

In [ ]:
## Create a dataframe

df = pd.DataFrame(dict(x=X_dirty[:,0] , y=X_dirty[:,1], label= 1 - y_dirty))
print(df.columns[0])
colors = {0 : 'red', 1:'blue'}
df_grouped = df.groupby('label')

In [ ]:
# Graphique 
fig , ax = plt.subplots()
for key, group in df_grouped:
    group.plot(ax=ax , kind = 'scatter', x='x', y='y', label=key, color=colors[key])

plt.grid()
plt.title("Topologie du jeu de données d'apprentissage dans R^2")
plt.show()


Ce qu'on remarque c'est la forte densité de probabilité sur la droite d'équation : y = 0  , que la densité de probabilité est très forte pour les éléments de la classe 0.

(à appronfondir)

# Modèles génératifs QDA et effondrement paramétrique

In [ ]:
## Entrainement des données
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# On crée nos datasets de train et de test. => On prend 80% train 20% test
X_train, X_test, y_train, y_test = train_test_split(X_dirty, y_dirty, test_size=0.2, random_state=42)

# Analyse quadratique discriminante
qda_clf = QuadraticDiscriminantAnalysis()

qda_clf.fit(X_train, y_train)


In [ ]:
# Plotting the boundaries
disp = DecisionBoundaryDisplay.from_estimator(qda_clf, 
                                              X_train, 
                                              response_method="predict",
                                              xlabel=df.columns[0], ylabel=df.columns[1],
                                              alpha=0.5, 
                                              cmap=plt.cm.coolwarm)
# Plotting the data points    
disp.ax_.scatter(X_train[:, 0], X_train[:, 1], 
                 c=y_train, edgecolor="k",
                 cmap=plt.cm.coolwarm)

plt.title("Test")
plt.xlim(right=3)
plt.show()

In [ ]:
# Create a mesh to plot the decision boundaries
x_min, x_max = (-3 , 3)
y_min, y_max = (-3, 3)
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                     np.arange(y_min, y_max, 0.01))

# Predict class using the classifier for each point in the mesh
Z = qda_clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot the decision boundaries
plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.Paired)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors='k', marker='o', label='Training Data')
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, edgecolors='k', marker='s', label='Test Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.xlim(right=3)
plt.title('QDA Decision Boundaries')
plt.legend()
plt.show()

Le fait qu'il y ait des outliers pour la classe 0 biaise totalement notre séparateur qui nous renvoit des frontières sorties du chapeau ...

La formule de la distance de Mahalanobis : $$ D_{M}=\sqrt{(X - \mu)^T\hat{\Sigma}^{-1}(X - \mu)} $$

D'après la formule de la covariance : $$ \hat{\Sigma} = 1/(N_{k} - 1) \sum (X - \hat{\mu}_{k}) (X - \hat{\mu}_{k})^{T} $$

On remarque que les espérances font parties de du calcul de la distance de Mahalanobis et que l'espérance est très sensible aux valeurs extrêmes (les outliers) , c'est pourquoi, une fois le calcul des frontières, il y a des petits problèmes ...

## Analyse LSA

L'analyse discriminante linéaire (LSA) n'est pas adaptée car les données ne peuvent être séparées linéairement quand bien même il n'y aurait pas d'outliers.

## Utilisation IsolationForest

Isolation Forest est l'un des algorithmes les plus populaires dans la detection d'anomalie , on parle de 'foret' car chaque élément du dataset se voit appliquer des décisions si oui ou non il est catégoriser outlier. 
Complexité linéaire, très bien 

In [ ]:
#Paramètres par défault trouvé sur notebook 
Iforest = IsolationForest(max_samples=100, 
                          random_state=1111,
                         contamination=0.05,
                         max_features=1.0,
                         n_estimators=100,
                         verbose=1,
                         n_jobs=-1)
Iforest.fit(X_dirty)

In [ ]:
predictions_outliers= Iforest.predict(X_dirty)

#print(predictions_outliers)
predi_adjusted = [1 if x == -1 else 0 for x in predictions_outliers]
condition_true = np.array(predi_adjusted) == 0
print(condition_true)
X_cleaned = np.array(X_dirty)[condition_true]
y_cleaned = np.array(y_dirty)[condition_true]

In [ ]:
X_cleaned
y_cleaned


In [ ]:
## Create a dataframe cleaned

df_cleaned = pd.DataFrame(dict(x=X_cleaned[:,0] , y=X_cleaned[:,1], label= 1 - y_cleaned))
colors = {0 : 'red', 1:'blue'}
df_grouped_cleaned = df_cleaned.groupby('label')



In [ ]:
# Graphique 
fig , ax = plt.subplots()
for key, group in df_grouped_cleaned:
    group.plot(ax=ax , kind = 'scatter', x='x', y='y', label=key, color=colors[key])

plt.grid()
plt.title("Topologie du jeu de données d'apprentissage dans R^2")
plt.show()

In [ ]:
## Entrainement des données
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

# On crée nos datasets de train et de test. => On prend 80% train 20% test
X_train, X_test, y_train, y_test = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)

# Analyse quadratique discriminante
qda_clf_cleaned = QuadraticDiscriminantAnalysis()

qda_clf_cleaned.fit(X_train, y_train)

# Plotting the boundaries
disp = DecisionBoundaryDisplay.from_estimator(qda_clf_cleaned, 
                                              X_train, 
                                              response_method="predict",
                                              xlabel=df_cleaned.columns[0], ylabel=df_cleaned.columns[1],
                                              alpha=0.5, 
                                              cmap=plt.cm.coolwarm)
# Plotting the data points    
disp.ax_.scatter(X_train[:, 0], X_train[:, 1], 
                 c=y_train, edgecolor="k",
                 cmap=plt.cm.coolwarm)

plt.title("Separator Area with QDA")
plt.xlim(right=3)
plt.show()

## K - NN , Topologie locale

In [ ]:
## Entrainement model 
X_train, X_test, y_train, y_test = train_test_split(X_dirty, y_dirty, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)



In [ ]:
def classify_and_plot(X, y):
    ''' 
    split data, fit, classify, plot and evaluate results 
    '''
    # split data into training and testing set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 41)

    # init vars
    n_neighbors = 5
    h           = .02  # step size in the mesh

    # Create color maps
    cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
    cmap_bold  = ListedColormap(['#FF0000', '#0000FF'])

    rcParams['figure.figsize'] = 5, 5
    for weights in ['uniform', 'distance']:
        # we create an instance of Neighbours Classifier and fit the data.
        clf = neighbors.KNeighborsClassifier(n_neighbors, weights=weights)
        clf.fit(X_train, y_train)

        # Plot the decision boundary. For that, we will assign a color to each
        # point in the mesh [x_min, x_max]x[y_min, y_max].
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                             np.arange(y_min, y_max, h))
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])

        # Put the result into a color plot
        Z = Z.reshape(xx.shape)
        fig = plt.figure()
        plt.pcolormesh(xx, yy, Z, cmap=cmap_light)

        # Plot also the training points, x-axis = 'Glucose', y-axis = "BMI"
        plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor='k', s=20)   
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        plt.title("0/1 outcome classification (k = %i, weights = '%s')" % (n_neighbors, weights))
        plt.show()

        # evaluate
        y_expected  = y_test
        y_predicted = clf.predict(X_test)

        # print results
        print('----------------------------------------------------------------------')
        print('Classification report')
        print('----------------------------------------------------------------------')
        print('\n', classification_report(y_expected, y_predicted))


In [ ]:
classify_and_plot(X_dirty, y_dirty)

Le code que j'ai récupéré ici : 'https://stackoverflow.com/questions/56153726/plot-k-nearest-neighbor-graph-with-8-features' utilise les deux manières de calculer les poids suivant la distance ou de manière uniforme.

On remarque que le KNN ne subit pas la même déformation que QDA malgré les outliers dans la région [-3, 3]
On se place dans le plan affine $$ R^{2} $$

 "Soit S un ensemble fini de n points du plan ; les éléments de S sont appelés centres, sites ou encore germes. On appelle région de Voronoï — ou cellule de Voronoï — associée à un germe p de S, l’ensemble des points qui sont plus proches de p que de tout autre point de S" (Source wikipédia)
Ainsi les cellules de Voronoï des outiliers vont n'avoir aucune influence localement sur notre région [-3 , 3] car bloquée par la zone d'influence des autres éléments ayant le label 1

In [ ]:
## Pipeline d'étapes

pipeline= Pipeline([ ('scaler', MinMaxScaler()),('knn',neighbors.KNeighborsClassifier(n_neighbors=5))])

pipeline.fit(X_dirty, y_dirty)

# Plotting the boundaries
disp = DecisionBoundaryDisplay.from_estimator(pipeline, 
                                              X_dirty, 
                                              response_method="predict",
                                              xlabel=df_cleaned.columns[0], ylabel=df_cleaned.columns[1],
                                              alpha=0.5, 
                                              cmap=plt.cm.coolwarm)
# Plotting the data points    
disp.ax_.scatter(X_dirty[:, 0], X_dirty[:, 1], 
                 c=y_dirty, edgecolor="k",
                 cmap=plt.cm.coolwarm)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Separator Area Normalisé & KNN region [-0.1, 1.1]")
plt.xlim((0, 0.15))
plt.ylim((-0.1, 1.1))
plt.show()

Formule du MinMaxScaler : 

$$ x_scaled = x - x_min / x_max - x_min $$

Les outliers avec des valeurs abberantes sur x1 "rétrécissent" encore plus les données ce qui fait que la distance entre les points est plus faible.